This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of Quickstart on link https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


# Compare the Outputs of Pytorch Model and QONNX Model

In [7]:
import torch
import models.QFastSCNN as qfscnn
from my_finn_utils import load_state_dict
from config import CROP_SIZE, NUM_CLASSES

pytorch_model = qfscnn.QFastSCNN(NUM_CLASSES)
pytorch_model = load_state_dict(pytorch_model, path="../train_environment/model_weights/quant_params/best_quant_model.pth", strict=True)
pytorch_model.eval()

dummy_input = torch.randn(1, 3, *CROP_SIZE)
dummy_input.size()

Carregando modelo best_quant_model


torch.Size([1, 3, 768, 768])

In [10]:
# Run a foward pass on Pytorch model
with torch.inference_mode():
    pytorch_output = pytorch_model(dummy_input)
pytorch_output.size()

/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:195: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.cat([x, feat1, feat2, feat3, feat4], dim=1)


torch.Size([1, 19, 96, 96])